In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del carpeta "production_country"

###Paso 1 - Leer los archivos JSON usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

In [0]:
productions_countries_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("countryId", IntegerType(), False)
])

In [0]:
productions_countries_df = spark.read\
    .option("multiLine", True)\
    .schema(productions_countries_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/production_country")

In [0]:
productions_countries_df.count()

3750

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import lit, current_timestamp

In [0]:
productions_countries_final_df = add_ingestion_date(productions_countries_df)\
    .withColumnsRenamed({"movieId": "movie_id",
                         "countryId": "country_id"})\
    .withColumn("environment", lit("Production"))\
    .withColumn("file_date", lit(v_file_date))

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#overwrite_partition("movie_silver", "productions_countries", "file_date", v_file_date)

In [0]:
#productions_countries_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/productions_countries")

In [0]:
#productions_countries_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.productions_countries")

condition_merge = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "productions_countries", productions_countries_final_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.productions_countries
GROUP BY file_date;

file_date,count(1)
2024-12-16,3750
2024-12-23,1250
2024-12-30,1436


In [0]:
dbutils.notebook.exit("Exitoso")